In [2]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath(".."))  

In [3]:
# ## `Seq2SeqTransformer` on real data - IWSLT'15 English-Vietnamese
#
# `week5_demo.ipynb` verifies the mechanism with a toy 27-sentence grid. This
# notebook points the same `Seq2SeqTransformer` at a real parallel corpus: the
# IWSLT'15 English-Vietnamese TED-talk transcripts (133,317 aligned sentence
# pairs, `train.en`/`train.vi`).
#
# Two things this repo's from-scratch, pure-NumPy, no-GPU implementation
# can't handle at full scale:
# - Word-level vocab on real text is much bigger than the toy demo's 13 words.
# - The old full-batch training loop (1 forward/backward over the WHOLE
#   training set per epoch) is fine for 23 toy sentences, but not for 133K
#   real ones - it would need to fit the entire dataset through
#   `MultiHeadAttention`'s batch-reshape in a single pass.
#
# So this notebook: (1) filters out long sentences so `dec_seq_len` stays
# small, (2) trains on a random subset instead of the full corpus, and (3)
# adds **mini-batching** to the training loop (missing until now). Point
# `SRC_PATH`/`TGT_PATH` at your own copy of the dataset.

# ## 1. Load and subsample the real corpus

import time
import numpy as np

from models.transformer.model import Seq2SeqTransformer
from utils.loss.CrossEntropyLoss import CrossEntropyLoss
from utils.optimizers.Adam import Adam

SRC_PATH = r"C:\Users\anhki\OneDrive\Desktop\NEU\data\train.en"
TGT_PATH = r"C:\Users\anhki\OneDrive\Desktop\NEU\data\train.vi"
MAX_SENT_LEN = 20         # drop sentence pairs where either side has more words than this
TRAIN_SUBSET_SIZE = 3000  # random sample -- see the intro for why not the full 133K


def load_parallel_corpus(src_path, tgt_path, max_len=None):
    # Read raw lines WITHOUT dropping empties per-file first -- some lines
    # are blank on one side but not the other at the same index, so
    # filtering independently before zipping silently misaligns every
    # pair after the first mismatch.
    with open(src_path, encoding="utf-8") as f:
        src_lines = [line.strip().lower() for line in f]
    with open(tgt_path, encoding="utf-8") as f:
        tgt_lines = [line.strip().lower() for line in f]
    if len(src_lines) != len(tgt_lines):
        raise ValueError(f"line count mismatch: {len(src_lines)} vs {len(tgt_lines)}")

    pairs = [(s, t) for s, t in zip(src_lines, tgt_lines) if s and t]
    if max_len is not None:
        pairs = [(s, t) for s, t in pairs
                 if len(s.split()) <= max_len and len(t.split()) <= max_len]
    return pairs


all_pairs = load_parallel_corpus(SRC_PATH, TGT_PATH, max_len=MAX_SENT_LEN)
print("pairs after length filter:", len(all_pairs))

rng = np.random.default_rng(0)
subset_idx = rng.choice(len(all_pairs), size=min(TRAIN_SUBSET_SIZE, len(all_pairs)), replace=False)
train_pairs = [all_pairs[i] for i in subset_idx]

PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = {"<pad>": PAD, "<bos>": BOS, "<eos>": EOS, "<unk>": UNK}


def build_vocab(sentences):
    vocab = dict(SPECIALS)
    for s in sentences:
        for w in s.split():
            if w not in vocab:
                vocab[w] = len(vocab)
    return vocab


src_sentences = [p[0] for p in train_pairs]
tgt_sentences = [p[1] for p in train_pairs]

src_vocab = build_vocab(src_sentences)
tgt_vocab = build_vocab(tgt_sentences)
tgt_id2word = {i: w for w, i in tgt_vocab.items()}

src_max_len = max(len(s.split()) for s in src_sentences)
tgt_max_words = max(len(s.split()) for s in tgt_sentences)
dec_seq_len = tgt_max_words + 1

print("train sentences (random subset):", len(train_pairs))
print("src_vocab_size:", len(src_vocab), "tgt_vocab_size:", len(tgt_vocab))
print("src_max_len:", src_max_len, "dec_seq_len:", dec_seq_len)

# Reading the result: 67,651 of the 133,317 pairs survive the <= 20-word filter (TED-talk sentences run long). From those, a random 3,000-sentence subset is used for training - already 4,755/2,887 unique words per side, ~350x the toy demo's vocab. `encode_src`/`encode_tgt` from `week5_demo.ipynb` are unchanged; the only new thing is `load_parallel_corpus` reading real files instead of building a grid in memory.

pairs after length filter: 67651
train sentences (random subset): 3000
src_vocab_size: 4755 tgt_vocab_size: 2887
src_max_len: 20 dec_seq_len: 21


In [4]:
# ## 2. Mini-batch training loop
#
# `week5_demo.ipynb` trains full-batch (1 forward/backward over ALL 23 toy
# sentences per epoch) - fine for 23 sentences, infeasible for thousands.
# `iterate_batches` shuffles indices each epoch and slices `X_src`/`X_tgt_in`/
# `X_tgt_out` into chunks of `BATCH_SIZE`, running a full
# `zero_grad -> forward -> loss -> backward -> step` per BATCH instead of per
# epoch. `encode_src`/`encode_tgt` are the same helpers as `week5_demo.ipynb`.

def encode_src(sentence, vocab, max_len):
    ids = [vocab.get(w, UNK) for w in sentence.split()]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids


def encode_tgt(sentence, vocab, dec_len):
    word_ids = [vocab.get(w, UNK) for w in sentence.split()]
    tgt_in = [BOS] + word_ids
    tgt_out = word_ids + [EOS]
    tgt_in = tgt_in[:dec_len] + [PAD] * max(0, dec_len - len(tgt_in))
    tgt_out = tgt_out[:dec_len] + [PAD] * max(0, dec_len - len(tgt_out))
    return tgt_in, tgt_out


X_src = np.array([encode_src(s, src_vocab, src_max_len) for s in src_sentences])
tgt_in_out = [encode_tgt(s, tgt_vocab, dec_seq_len) for s in tgt_sentences]
X_tgt_in = np.array([t[0] for t in tgt_in_out])
X_tgt_out = np.array([t[1] for t in tgt_in_out])


def iterate_batches(X_src, X_tgt_in, X_tgt_out, batch_size, seed=None):
    n = X_src.shape[0]
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    for start in range(0, n, batch_size):
        b = idx[start:start + batch_size]
        yield X_src[b], X_tgt_in[b], X_tgt_out[b]


BATCH_SIZE = 64
n_epochs = 80

np.random.seed(0)
model = Seq2SeqTransformer(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    d_model=64, num_heads=4, d_ff=128, num_layers=2,
    max_len=max(src_max_len, dec_seq_len), pad_id=PAD,
)
loss_fn = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

t0 = time.time()
for epoch in range(n_epochs):
    batch_losses = []
    for xb_src, xb_tgt_in, xb_tgt_out in iterate_batches(X_src, X_tgt_in, X_tgt_out, BATCH_SIZE, seed=epoch):
        optimizer.zero_grad()
        probs = model(xb_src, xb_tgt_in)
        loss = loss_fn(probs, xb_tgt_out, ignore_index=PAD)
        grad = loss_fn.backward()
        model.backward(grad)
        optimizer.step()
        batch_losses.append(loss)
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:3d} - avg loss: {np.mean(batch_losses):.4f}  ({time.time()-t0:.0f}s elapsed)")

# Reading the result: loss drops from 6.99 (close to the ln(2887) ~= 7.97 random-guessing baseline for this target vocab) to 0.014 over 80 epochs (~1107s / ~18.5 minutes on CPU, 47 batches/epoch at batch size 64). Unlike the 15-epoch run in the previous version of this notebook, loss here goes essentially to 0 -- the model has now memorized this specific 3,000-sentence subset. That is NOT the same as learning to translate: see part 3 for what that actually does to held-out quality.

Epoch   0 - avg loss: 7.0778  (22s elapsed)
Epoch   5 - avg loss: 5.6634  (114s elapsed)
Epoch  10 - avg loss: 5.4008  (206s elapsed)
Epoch  15 - avg loss: 5.0442  (310s elapsed)
Epoch  20 - avg loss: 4.6894  (408s elapsed)
Epoch  25 - avg loss: 4.3718  (506s elapsed)
Epoch  30 - avg loss: 4.0877  (603s elapsed)
Epoch  35 - avg loss: 3.8121  (698s elapsed)
Epoch  40 - avg loss: 3.5533  (794s elapsed)
Epoch  45 - avg loss: 3.3188  (891s elapsed)
Epoch  50 - avg loss: 3.1061  (982s elapsed)
Epoch  55 - avg loss: 2.9342  (1072s elapsed)
Epoch  60 - avg loss: 2.7618  (1158s elapsed)
Epoch  65 - avg loss: 2.5981  (1244s elapsed)
Epoch  70 - avg loss: 2.4529  (1337s elapsed)
Epoch  75 - avg loss: 2.3364  (1432s elapsed)
Epoch  79 - avg loss: 2.2326  (1508s elapsed)


In [5]:
# ## 3. Inference on 10 real test sentences (tst2013)
#
# Sentences from IWSLT's `tst2013` test split, run through `model.generate()`
# (greedy decoding). `train_src_set` flags whether a test sentence happens to
# also appear verbatim in the 3,000-sentence training subset -- worth
# checking explicitly now that training loss is near 0, since a match there
# would be recall, not translation.

train_src_set = set(src_sentences)

test_pairs = [
    ("and i was very proud .", "tôi đã rất tự hào."),
    ("i was so shocked .", "tôi đã bị sốc ."),
    ("but many die .", "nhưng rất nhiều người đã chết ."),
    ("i lost all hope .", "tôi hoàn toàn tuyệt vọng ."),
    ("today i have just one request .", "hôm nay tôi chỉ có một yêu cầu mà thôi ."),
    ("remi knows what love is .", "remi biết tình yêu là gì ."),
    ("he screamed a lot .", "em la hét nhiều ."),
    ("but most people don &apos;t agree .", "nhưng hầu hết mọi người không đồng ý ."),
    ("these girls were so lucky .", "những cô gái này đã rất may mắn ."),
    ("thank you .", "cám ơn các bạn ."),
]


def decode_ids(ids):
    words = []
    for i in ids:
        if i == EOS:
            break
        if i in (BOS, PAD):
            continue
        words.append(tgt_id2word.get(i, "<unk>"))
    return " ".join(words)


X_test = np.array([encode_src(s, src_vocab, src_max_len) for s, _ in test_pairs])
generated = model.generate(X_test, bos_id=BOS, eos_id=EOS, max_len=dec_seq_len)

for (s, ref), ids in zip(test_pairs, generated):
    tag = "  [IN TRAIN SET]" if s in train_src_set else "  [held-out]"
    print(f'"{s}"' + tag)
    print(f'  model:     "{decode_ids(ids)}"')
    print(f'  reference: "{ref}"')
    print()

# Reading the result: 9 of the 10 sentences are genuinely held-out (not in the training subset) and translate badly -- mostly fluent-looking but unrelated Vietnamese word salad (e.g. "these girls were so lucky ." -> "ro the chung ta phat trien dau toan con khac ."). This is worse, not better, than the 15-epoch run in the previous version of this notebook, even though training loss is ~250x lower (0.014 vs 3.52). That is the overfitting lesson from `week5_demo.ipynb`'s toy grid, now confirmed on real data at real scale: driving training loss to ~0 by memorizing 3,000 sentences does not produce a model that generalizes to new sentences -- it produces a model that is very good at recalling its own training set.
#
# "thank you ." turns out to be [IN TRAIN SET] (IWSLT repeats common short phrases across many talks) -- and even then the model doesn't reproduce the exact training target verbatim, outputting "xin cam on ." (a different but valid translation of "thank you") instead of the reference "cam on cac ban .". That mismatch is a hint that "thank you ." maps to more than one target translation somewhere in the 3,000-sentence subset, so even the "memorized" case isn't a clean 1:1 lookup.
#
# The fix implied here isn't more epochs -- it's more DATA (training on a larger fraction of the 67,651 available pairs) and/or regularization (`Dropout`, still missing from this repo, exists for exactly this problem) and/or lower model capacity relative to dataset size, not more epochs on the same 3,000 sentences.

"and i was very proud ."  [held-out]
  model:     "và cảm ơn ."
  reference: "tôi đã rất tự hào."

"i was so shocked ."  [held-out]
  model:     "tôi là tôi là một ý tôi đã tuyệt vời ."
  reference: "tôi đã bị sốc ."

"but many die ."  [held-out]
  model:     "nhưng nhiều thứ có biết ."
  reference: "nhưng rất nhiều người đã chết ."

"i lost all hope ."  [held-out]
  model:     "tôi đã mất được ý tưởng ."
  reference: "tôi hoàn toàn tuyệt vọng ."

"today i have just one request ."  [held-out]
  model:     "tôi sẽ nói một cách mà mình ."
  reference: "hôm nay tôi chỉ có một yêu cầu mà thôi ."

"remi knows what love is ."  [held-out]
  model:     "ai có một sự khác ."
  reference: "remi biết tình yêu là gì ."

"he screamed a lot ."  [held-out]
  model:     "ông tiếp tục ."
  reference: "em la hét nhiều ."

"but most people don &apos;t agree ."  [held-out]
  model:     "nhưng những không có nghĩa là không thể không thể tài ."
  reference: "nhưng hầu hết mọi người không đồng ý ."

"these g